# Olist E-Commerce Analysis-Ready Dataset

## Objective

The purpose of this notebook is to combine the cleaned Olist datasets into a reusable analysis-ready table for business analysis, SQL queries, and dashboard development.

The primary analytical grain will be one row per order item, uniquely represented by the combination of `order_id` and `order_item_id`. Cleaned Orders, Customers, Products, Sellers, Payments, and Reviews data will be joined carefully while validating row counts and preventing unintended row multiplication.

Only delivered orders will be used for completed-sales metrics. Data-quality exception flags will be retained so records can be included or excluded according to the requirements of each analysis.


## 1. Environment Setup and Processed Data Loading

The required Python libraries and processed-data path will be initialized. Each cleaned dataset will then be loaded independently from the processed-data directory to ensure this notebook does not depend on variables created in the cleaning notebook.


In [1]:
# Import required libraries and define the processed-data path
import numpy as np
import pandas as pd 
from pathlib import Path

Processed_Data_Path = Path("../data/processed")

print(Processed_Data_Path.resolve())


C:\Users\Hp\DataAnalyst\data\processed


In [2]:
import pyarrow as pa
print(pa.__version__)

25.0.1


### 1.1 Load the Cleaned Datasets

The cleaned datasets will be loaded from the processed-data directory. These files contain standardized fields and data-quality flags created during the cleaning phase.


In [3]:
# Load the cleaned datasets required for the analysis-ready table
orders = pd.read_csv(Processed_Data_Path/"olist_orders_clean.csv")
customers = pd.read_csv(Processed_Data_Path/"olist_customers_clean.csv")
order_items = pd.read_csv(Processed_Data_Path/"olist_order_items_clean.csv")
payments = pd.read_csv(Processed_Data_Path/"olist_order_payments_clean.csv")
reviews = pd.read_csv(Processed_Data_Path/"olist_order_reviews_clean.csv")
products = pd.read_csv(Processed_Data_Path/"olist_products_clean.csv")
sellers = pd.read_csv(Processed_Data_Path/"olist_sellers_clean.csv")

In [4]:
# Inspect data types after reloading the cleaned CSV files

orders.dtypes

order_id                                       str
customer_id                                    str
order_status                                   str
order_purchase_timestamp                       str
order_approved_at                              str
order_delivered_carrier_date                   str
order_delivered_customer_date                  str
order_estimated_delivery_date                  str
flag_delivered_missing_customer_date          bool
flag_delivered_missing_carrier_date           bool
flag_delivered_missing_approval_date          bool
flag_delivered_missing_critical_timestamp     bool
critical_timestamp_flag_count                int64
flag_carrier_before_purchase                  bool
flag_customer_before_carrier                  bool
flag_carrier_before_approval                  bool
flag_lifecycle_timestamp_exception            bool
dtype: object

In [5]:
customers.dtypes

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [6]:
order_items.dtypes

order_id                               str
order_item_id                        int64
product_id                             str
seller_id                              str
shipping_limit_date                    str
price                              float64
freight_value                      float64
flag_future_shipping_limit_date       bool
dtype: object

In [7]:
payments.dtypes

order_id                                 str
payment_sequential                     int64
payment_type                             str
payment_installments                   int64
payment_value                        float64
flag_payment_sequence_mismatch          bool
flag_payment_type_not_defined           bool
flag_invalid_payment_installments       bool
flag_zero_payment_value                 bool
flag_payment_quality_exception          bool
dtype: object

In [8]:
reviews.dtypes

review_id                              str
order_id                               str
review_score                         int64
review_comment_title                   str
review_comment_message                 str
review_creation_date                   str
review_answer_timestamp                str
review_response_duration               str
flag_review_response_over_30_days     bool
dtype: object

In [9]:
products.dtypes

product_id                              str
product_category_name                   str
product_name_length                 float64
product_description_length          float64
product_photos_qty                  float64
product_weight_g                    float64
product_length_cm                   float64
product_height_cm                   float64
product_width_cm                    float64
flag_missing_catalog_attributes        bool
flag_missing_physical_attributes       bool
flag_zero_product_weight               bool
flag_untranslated_category             bool
product_category_name_english           str
product_category_name_analysis          str
dtype: object

In [10]:
sellers.dtypes

seller_id                               str
seller_zip_code_prefix                int64
seller_city                             str
seller_state                            str
geolocation_reference_state             str
flag_missing_geolocation_reference     bool
flag_seller_state_mismatch             bool
flag_seller_location_exception         bool
dtype: object

### 1.2 Restore Analysis Data Types

CSV files do not preserve pandas-specific data types. After loading the processed files, timestamp columns, duration fields, and ZIP-code prefixes must be restored to their analysis-ready formats before the datasets are joined.


In [11]:
# Restore the cleaned Orders timestamp columns to datetime format
orders_timestamp_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

for cols in orders_timestamp_columns:
    orders[cols] = pd.to_datetime(orders[cols], errors= 'coerce')

# Confirm that all Orders timestamp columns use a datetime data type
orders[orders_timestamp_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [12]:
# Restore the cleaned Order Items shipping-limit date to datetime format
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors = 'coerce')

# Confirm that the Order Items shipping-limit date uses a datetime data type
order_items['shipping_limit_date'].dtype

dtype('<M8[us]')

In [13]:
# Restore the cleaned Reviews timestamp columns to datetime format
reviews_timestamp_columns = ['review_creation_date', 'review_answer_timestamp']

for cols in reviews_timestamp_columns:
    reviews[cols] = pd.to_datetime(reviews[cols], errors= 'coerce')

# Confirm that all Reviews timestamp columns use a datetime data type
reviews[reviews_timestamp_columns].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [14]:
# Restore the review response duration column to a timedelta data type
reviews['review_response_duration'] = pd.to_timedelta(reviews['review_response_duration'], errors = 'coerce')

# Confirm that the review response duration uses a timedelta data type
reviews['review_response_duration'].dtype

dtype('<m8[us]')

In [15]:
# Restore customer and seller ZIP-code prefixes as five-character strings
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].str.zfill(5)

sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].str.zfill(5)

In [16]:
# Confirm that customer and seller ZIP-code prefixes contain exactly five characters
print(f'Minimum length of customers zip code_prefix: {min(customers['customer_zip_code_prefix'].str.len())}')
print(f'Maximum length of customers zip code_prefix: {max(customers['customer_zip_code_prefix'].str.len())}')

print(f'Minimum length of sellers zip code_prefix: {min(sellers['seller_zip_code_prefix'].str.len())}')
print(f'Maximum length of sellers zip code_prefix: {max(sellers['seller_zip_code_prefix'].str.len())}')

Minimum length of customers zip code_prefix: 5
Maximum length of customers zip code_prefix: 5
Minimum length of sellers zip code_prefix: 5
Maximum length of sellers zip code_prefix: 5


In [17]:
# Verify that Orders timestamp conversion introduced no additional missing values

orders[orders_timestamp_columns].isna().sum()

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [18]:
# Verify that the Order Items shipping-limit date conversion introduced no missing values
order_items['shipping_limit_date'].isna().sum()

np.int64(0)

In [19]:
# Verify that Reviews timestamp and duration conversions introduced no missing values
reviews[['review_creation_date', 'review_answer_timestamp', 'review_response_duration']].isna().sum()

review_creation_date        0
review_answer_timestamp     0
review_response_duration    0
dtype: int64

## 2. Define and Validate the Analysis Grain

The analysis-ready dataset will use one row per order item, uniquely identified by the combination of `order_id` and `order_item_id`. The Order Items table will therefore serve as the base table for constructing the dataset.


In [20]:
# Validate the uniqueness of the order_id–order_item_id analysis grain
order_items[['order_id', 'order_item_id']].duplicated().sum()

np.int64(0)

In [21]:
# Record the baseline number of order-item rows before joining other tables
baseline_order_item_rows = order_items.shape[0]

In [22]:
baseline_order_item_rows

112650

## 3. Construct the Order-Item Analysis Base

The Order Items table serves as the base because the analysis-ready dataset uses one row per order item. Order-level, customer, product, and seller attributes will be added using many-to-one joins while preserving the original order-item row count.

### 3.1 Join Orders with Order Items


In [23]:
# Create the order-item-level base for the analysis-ready dataset
analysis_base = order_items.copy()

In [24]:
#compare analysis_base shape with order_items
if order_items.shape == analysis_base.shape:
    print("order_items and analysis_base have the same shape")
    print(f"order_items: {order_items.shape}")
    print(f"analysis_base: {analysis_base.shape}")
else:
    print("order_items and analysis_base have different shapes")
    print(f"order_items: {order_items.shape}")
    print(f"analysis_base: {analysis_base.shape}")

order_items and analysis_base have the same shape
order_items: (112650, 8)
analysis_base: (112650, 8)


In [25]:
#validate the Orders join key by checking:
#Missing order_id values
#Duplicated order_id values
print(f"Missing order_id values in Orders: {int(orders['order_id'].isna().sum())}")
print(f"Duplicated order_id values in Orders: {int(orders['order_id'].duplicated().sum())}")

Missing order_id values in Orders: 0
Duplicated order_id values in Orders: 0


In [26]:
# Add order-level attributes while preserving every order-item record
analysis_base = pd.merge(analysis_base, orders, on='order_id', how='left', validate='many_to_one')

In [27]:
#Validate that the Orders join preserved the order-item grain and matched every record.

print(f"New shape of analysis_base: {analysis_base.shape}")


New shape of analysis_base: (112650, 24)


In [28]:
if analysis_base.shape[0] == baseline_order_item_rows:
    print(f"both baseline_order_item_rows and new analysis_base rows are same: {baseline_order_item_rows}")
else:
    print(f"both baseline_order_item_rows and new analysis_base roes are different")
    print(f"New analysis_base rows: {analysis_base.shape[0]}")
    print(f"baseline_order_item_rows: {baseline_order_item_rows}")


both baseline_order_item_rows and new analysis_base rows are same: 112650


In [29]:
analysis_base['customer_id'].isna().sum()

np.int64(0)

### 3.2 Add Customer Attributes


In [30]:
#validate that customers['customer_id'] has no missing and duplicated values

print(f"Missing customer_id in Customers: {int(customers['customer_id'].isna().sum())}")
print(f"Number of duplicate customer_id in Customers: {int(customers['customer_id'].duplicated().sum())}")

Missing customer_id in Customers: 0
Number of duplicate customer_id in Customers: 0


In [31]:
#Add customer attributes while preserving every order-item record.
analysis_base = pd.merge(analysis_base, customers, on='customer_id', how='left', validate='many_to_one')

In [32]:
# Confirm that every order-item record matched a customer
analysis_base.shape

(112650, 28)

In [33]:
#validate this join by checking:Whether its row count remains 112,650
if analysis_base.shape[0] == baseline_order_item_rows:
    print(f"both baseline_order_item_rows and new analysis_base rows are same: {baseline_order_item_rows}")
else:
    print(f"both baseline_order_item_rows and new analysis_base roes are different")
    print(f"New analysis_base rows: {analysis_base.shape[0]}")
    print(f"baseline_order_item_rows: {baseline_order_item_rows}")

both baseline_order_item_rows and new analysis_base rows are same: 112650


In [34]:
# Confirm that every order-item record matched a customer
analysis_base['customer_unique_id'].isna().sum()

np.int64(0)

### 3.3 Add Product Attributes

In [35]:
#validate that products['product_id'] has no missing or duplicated values
print(f"Missing product_id in Products: {int(products['product_id'].isna().sum())}")
print(f"Number of duplicate product_id in Products: {int(products['product_id'].duplicated().sum())}")

Missing product_id in Products: 0
Number of duplicate product_id in Products: 0


In [36]:
#Add product and category attributes while preserving every order-item record.
analysis_base = pd.merge(analysis_base, products, on='product_id', how='left', validate='many_to_one')

In [37]:
#validate the new shape of analysis_base
analysis_base.shape

(112650, 42)

In [38]:
# Confirm that every order item received an analysis-ready product category
analysis_base['product_category_name_analysis'].isna().sum()

np.int64(0)

### 3.4 Add Seller Attributes


In [39]:
#Validate the completeness and uniqueness of the Sellers join key.
print(f"Missing seller_id in Sellers: {int(sellers['seller_id'].isna().sum())}")
print(f"Number of duplicate seller_id in Sellers: {int(sellers['seller_id'].duplicated().sum())}")

Missing seller_id in Sellers: 0
Number of duplicate seller_id in Sellers: 0


In [40]:
#Add seller and seller-location attributes while preserving every order-item record.
analysis_base = pd.merge(analysis_base, sellers, on='seller_id', how='left', validate='many_to_one')

In [41]:
#Validate that the Sellers join preserved the analysis grain and matched every order item.
analysis_base.shape

(112650, 49)

In [42]:
#Validate missing seller-city value 
analysis_base['seller_city'].isna().sum()

np.int64(0)

## 4. Aggregate One-to-Many Order Tables

Payments and Reviews can contain multiple records for a single order. Joining them directly to the order-item base would multiply rows and distort item counts, sales values, and other metrics. Therefore, each table must first be summarized to one row per `order_id`.

### 4.1 Aggregate Payment Records


In [43]:
#Validate the one-to-many relationship between Orders and Payments.
print(f"Total Payment rows: {payments.shape[0]}")
print(f"Distinct order_id count: {payments['order_id'].nunique()}")
print(f"Duplicated order_id occurrences: {int(payments['order_id'].duplicated().sum())}")

Total Payment rows: 103886
Distinct order_id count: 99440
Duplicated order_id occurrences: 4446


In [44]:
# Aggregate payment records to one row per order before joining
payments_order_summary = payments.groupby('order_id').agg(
    total_payment_value =('payment_value', 'sum'), 
    payment_record_count=('payment_value', 'count'),
    distinct_payment_type_count=('payment_type', 'nunique'),
    maximum_payment_installments=('payment_installments', 'max'),
    flag_payment_quality_exception=('flag_payment_quality_exception', 'any')
    ).reset_index()

payments_order_summary

,order_id,total_payment_value,payment_record_count,distinct_payment_type_count,maximum_payment_installments,flag_payment_quality_exception
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2,False
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3,False
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5,False
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3,False
...,...,...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,1,1,False
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,1,1,False
99437,fffce4705a9662cd70adb13d4a31832d,116.85,1,1,3,False
99438,fffe18544ffabc95dfada21779c9644f,64.71,1,1,3,False


In [45]:
#Validate the order-level payment summary before joining.
print(f"payments_order_summary shape: {payments_order_summary.shape}")
print(f"duplicated order_id in payments_order_summary: {int(payments_order_summary['order_id'].duplicated().sum())}")
print(f"Missing values across all summary columns: {(payments_order_summary.isna().sum()).sum()}")

payments_order_summary shape: (99440, 6)
duplicated order_id in payments_order_summary: 0
Missing values across all summary columns: 0


In [46]:
#Calculate the total value paid through each payment type within every order.
payment_type_value_per_order = payments.groupby(['order_id', 'payment_type'])['payment_value'].sum()
payment_type_value_per_order

order_id                          payment_type
00010242fe8c5a6d1ba2dd792cb16214  credit_card      72.19
00018f77f2f0320c557190d7a144bdd3  credit_card     259.83
000229ec398224ef6ca0657da4fc703e  credit_card     216.87
00024acbcdf0a6daa1e931b038114c75  credit_card      25.78
00042b26cf59d7ce69dfabb4e55b4fd9  credit_card     218.04
                                                   ...  
fffc94f6ce00a00581880bf54a75a037  boleto          343.40
fffcd46ef2263f404302a634eb57f7eb  boleto          386.53
fffce4705a9662cd70adb13d4a31832d  credit_card     116.85
fffe18544ffabc95dfada21779c9644f  credit_card      64.71
fffe41c64501cc87c801fd61db3f6244  credit_card      55.79
Name: payment_value, Length: 101686, dtype: float64

In [47]:
# Identify the dominant payment type for each order based on payment value
dominant_payment_type_index = (payment_type_value_per_order.groupby(level='order_id').idxmax())
dominant_payment_type_index.head()

order_id
00010242fe8c5a6d1ba2dd792cb16214    (00010242fe8c5a6d1ba2dd792cb16214, credit_card)
00018f77f2f0320c557190d7a144bdd3    (00018f77f2f0320c557190d7a144bdd3, credit_card)
000229ec398224ef6ca0657da4fc703e    (000229ec398224ef6ca0657da4fc703e, credit_card)
00024acbcdf0a6daa1e931b038114c75    (00024acbcdf0a6daa1e931b038114c75, credit_card)
00042b26cf59d7ce69dfabb4e55b4fd9    (00042b26cf59d7ce69dfabb4e55b4fd9, credit_card)
Name: payment_value, dtype: object

In [48]:
len(dominant_payment_type_index)

99440

In [49]:
#Extract the dominant payment type from each selected order–payment-type combination.
dominant_payment_type = dominant_payment_type_index.apply(lambda x: x[1])

In [50]:
#Add the dominant payment type to the order-level payment summary.
payments_order_summary['dominant_payment_type'] = payments_order_summary['order_id'].map(dominant_payment_type)
payments_order_summary.head()

,order_id,total_payment_value,payment_record_count,distinct_payment_type_count,maximum_payment_installments,flag_payment_quality_exception,dominant_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,1,2,False,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,1,3,False,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,1,5,False,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,1,2,False,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,1,3,False,credit_card


In [51]:
# Confirm that every payment summary record received a dominant payment type
payments_order_summary.isna().sum()

order_id                          0
total_payment_value               0
payment_record_count              0
distinct_payment_type_count       0
maximum_payment_installments      0
flag_payment_quality_exception    0
dominant_payment_type             0
dtype: int64

## 4. Aggregate and Add Order-Level Measures

Payments and Reviews can contain multiple records for the same order. These tables must be aggregated to one row per `order_id` before joining so that the order-item grain is preserved.

### 4.1 Create the Order-Level Payment Summary


In [52]:
#Add order-level payment measures while preserving every order-item record.
analysis_base = pd.merge(analysis_base, payments_order_summary, on='order_id', how='left', validate='many_to_one')

In [53]:
analysis_base.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,flag_future_shipping_limit_date,customer_id,order_status,...,geolocation_reference_state,flag_missing_geolocation_reference,flag_seller_state_mismatch,flag_seller_location_exception,total_payment_value,payment_record_count,distinct_payment_type_count,maximum_payment_installments,flag_payment_quality_exception,dominant_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,False,3ce436f183e68e07877b285a838db11a,delivered,...,RJ,False,True,True,72.19,1.0,1.0,2.0,False,credit_card
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,False,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,SP,False,False,False,259.83,1.0,1.0,3.0,False,credit_card
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,False,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,MG,False,False,False,216.87,1.0,1.0,5.0,False,credit_card
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,False,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,SP,False,False,False,25.78,1.0,1.0,2.0,False,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,False,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,PR,False,False,False,218.04,1.0,1.0,3.0,False,credit_card


In [54]:
#Validate that the Payment summary join preserved the analysis grain and identify unmatched orders.
print(f"The new shape of analysis_base: {analysis_base.shape}")
print(f"Missing total_payment_value values: {int(analysis_base['total_payment_value'].isna().sum())}")

missing_total_payment_value = analysis_base[analysis_base['total_payment_value'].isna()]
print(f"Number of distinct order_id values among those missing-payment rows: {missing_total_payment_value['order_id'].nunique()}")

The new shape of analysis_base: (112650, 55)
Missing total_payment_value values: 3
Number of distinct order_id values among those missing-payment rows: 1


In [55]:
missing_total_payment_value = analysis_base[analysis_base['total_payment_value'].isna()]
missing_total_payment_value

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,flag_future_shipping_limit_date,customer_id,order_status,...,geolocation_reference_state,flag_missing_geolocation_reference,flag_seller_state_mismatch,flag_seller_location_exception,total_payment_value,payment_record_count,distinct_payment_type_count,maximum_payment_installments,flag_payment_quality_exception,dominant_payment_type
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,False,86dc2ffce2dfff336de2f386a786e574,delivered,...,PR,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,False,86dc2ffce2dfff336de2f386a786e574,delivered,...,PR,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,False,86dc2ffce2dfff336de2f386a786e574,delivered,...,PR,False,False,False,NaN,NaN,NaN,NaN,NaN,NaN


### 4.2 Aggregate and Add Review Measures

The Reviews table may contain multiple review records for the same order. Review information will therefore be aggregated to one row per `order_id` before joining, preventing duplication of order-item records.


In [56]:
#Validate the one-to-many relationship between Orders and Reviews.
print(f"Total review rows: {len(reviews)}")
print(f"Distinct order_id count: {reviews['order_id'].nunique()}")
print(f"Duplicated order_id occurrences: {int(reviews['order_id'].duplicated().sum())}")

Total review rows: 99224
Distinct order_id count: 98673
Duplicated order_id occurrences: 551


In [57]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_response_duration,flag_review_response_over_30_days
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59,0 days 21:46:59,False
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13,1 days 03:05:13,False
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24,1 days 14:36:24,False
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06,0 days 22:02:06,False
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53,1 days 10:26:53,False


In [58]:
#Aggregate review records to one row per order before joining.
reviews_order_summary = reviews.groupby('order_id').agg(
    review_record_count = ('review_id', 'count'),
    average_review_score = ('review_score', 'mean'),
    review_title_count = ('review_comment_title', 'count'),
    review_message_count = ('review_comment_message', 'count'),
    average_review_response_duration = ('review_response_duration', 'mean'),
    flag_review_response_over_30_days = ('flag_review_response_over_30_days', 'any')
).reset_index()
reviews_order_summary.head()

,order_id,review_record_count,average_review_score,review_title_count,review_message_count,average_review_response_duration,flag_review_response_over_30_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,0,1,1 days 10:57:03,False
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,0,0,2 days 11:34:13,False
2,000229ec398224ef6ca0657da4fc703e,1,5.0,0,1,0 days 16:06:31,False
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,0,0,0 days 16:39:01,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,0,1,1 days 10:54:59,False


In [59]:
#Validate the order-level review summary before joining.

print(f"reviews_order_summary shape: {reviews_order_summary.shape}")
print(f"Duplicated order_id count: {int(reviews_order_summary['order_id'].duplicated().sum())}")
print(f"Missing values across all columns: {int((reviews_order_summary.isna().sum()).sum())}")

reviews_order_summary shape: (98673, 7)
Duplicated order_id count: 0
Missing values across all columns: 0


In [60]:
#Add order-level review measures while preserving every order-item record.
analysis_base = pd.merge(analysis_base, reviews_order_summary, on='order_id', how='left', validate='many_to_one')

In [61]:
#Validate that the Review summary join preserved the analysis grain and identify orders without reviews.
print(f"The new shape of analysis_base: {analysis_base.shape}")
print(f"Missing average_review_score values: {int(analysis_base['average_review_score'].isna().sum())}")

missing_average_review_score = analysis_base[analysis_base['average_review_score'].isna()]
print(f"Distinct order_id count among those missing-review rows :{missing_average_review_score['order_id'].nunique()}")

The new shape of analysis_base: (112650, 61)
Missing average_review_score values: 942
Distinct order_id count among those missing-review rows :749


## 5. Define the Delivered-Order Analysis Scope

The project’s primary business analysis focuses on completed sales. Therefore, the full joined order-item base is filtered to retain only orders whose `order_status` is `delivered`. Existing data-quality flags are retained so that records with timestamp exceptions can be excluded only from metrics that require those timestamps.


In [62]:
#Create the delivered-order-item dataset for the primary business analysis.
delivered_order_items_records = analysis_base[analysis_base['order_status']=='delivered']
delivered_order_items = delivered_order_items_records.copy()
delivered_order_items = delivered_order_items.reset_index(drop = True)
delivered_order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,flag_future_shipping_limit_date,customer_id,order_status,...,distinct_payment_type_count,maximum_payment_installments,flag_payment_quality_exception,dominant_payment_type,review_record_count,average_review_score,review_title_count,review_message_count,average_review_response_duration,flag_review_response_over_30_days
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,False,3ce436f183e68e07877b285a838db11a,delivered,...,1.0,2.0,False,credit_card,1.0,5.0,0.0,1.0,1 days 10:57:03,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,False,f6dd3ec061db4e3987629fe6b26e5cce,delivered,...,1.0,3.0,False,credit_card,1.0,4.0,0.0,0.0,2 days 11:34:13,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,False,6489ae5e4333f3693df5ad4372dab6d3,delivered,...,1.0,5.0,False,credit_card,1.0,5.0,0.0,1.0,0 days 16:06:31,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,False,d4eb9395c8c0431ee92fce09860c5a06,delivered,...,1.0,2.0,False,credit_card,1.0,4.0,0.0,0.0,0 days 16:39:01,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,False,58dbd0b2d70206bf40e62cd34e84d795,delivered,...,1.0,3.0,False,credit_card,1.0,5.0,0.0,1.0,1 days 10:54:59,False


In [63]:
#Validate the delivered-order-item scope and analysis grain.
print(f"Total number of rows: {delivered_order_items.shape[0]}")
print(f"Number of distinct order_id values: {delivered_order_items['order_id'].nunique()}")
print(f"Unique order_status values: {delivered_order_items['order_status'].unique()}")
print(f"Number of duplicated order_id-order_item_id combinations: {int(delivered_order_items[['order_id','order_item_id']].duplicated().sum())}")

Total number of rows: 110197
Number of distinct order_id values: 96478
Unique order_status values: <ArrowStringArray>
['delivered']
Length: 1, dtype: str


Number of duplicated order_id-order_item_id combinations: 0


## 6. Handle Join-Generated Missing Values

Left joins preserve all delivered order-item records, but orders without corresponding Payment or Review records produce missing values in the joined columns. These missing values will be handled according to their meaning without treating unavailable monetary values or review scores as zero.


In [64]:
#check missing values
missing_values = delivered_order_items.isna().sum()
missing_values[missing_values > 1]

order_approved_at                      15
order_delivered_carrier_date            2
order_delivered_customer_date           8
product_category_name                1537
product_name_length                  1537
product_description_length           1537
product_photos_qty                   1537
product_weight_g                       18
product_length_cm                      18
product_height_cm                      18
product_width_cm                       18
product_category_name_english        1559
geolocation_reference_state           249
total_payment_value                     3
payment_record_count                    3
distinct_payment_type_count             3
maximum_payment_installments            3
flag_payment_quality_exception          3
dominant_payment_type                   3
review_record_count                   827
average_review_score                  827
review_title_count                    827
review_message_count                  827
average_review_response_duration  

In [65]:
#Inspect missing Payment measures after constructing the delivered dataset.
payment_summary_columns = payments_order_summary.columns
delivered_order_items[payment_summary_columns].isna().sum()

order_id                          0
total_payment_value               3
payment_record_count              3
distinct_payment_type_count       3
maximum_payment_installments      3
flag_payment_quality_exception    3
dominant_payment_type             3
dtype: int64

In [66]:
#Flag delivered order-item rows whose order has no Payment record
flag_missing_payment_record = delivered_order_items['total_payment_value'].isna()
delivered_order_items['flag_missing_payment_record'] = flag_missing_payment_record
delivered_order_items['flag_missing_payment_record'].sum()

np.int64(3)

In [67]:
#Validate the missing-payment flag and affected order count.
print(f"The datatype of flag_missing_payment_record: {delivered_order_items['flag_missing_payment_record'].dtype}")
orders_flagged_missing_payment = delivered_order_items[delivered_order_items['flag_missing_payment_record']==True]
print(f"The number of distinct order_id values among flagged rows: {orders_flagged_missing_payment['order_id'].nunique()}")

The datatype of flag_missing_payment_record: bool
The number of distinct order_id values among flagged rows: 1


In [68]:
before_replacing_na_with_0 = delivered_order_items[['payment_record_count','distinct_payment_type_count']].isna().sum()
before_replacing_na_with_0

payment_record_count           3
distinct_payment_type_count    3
dtype: int64

In [69]:
# Set missing payment counts to zero and restore their integer data types
delivered_order_items[['payment_record_count','distinct_payment_type_count']] = delivered_order_items[['payment_record_count','distinct_payment_type_count']].fillna(0)
delivered_order_items[['payment_record_count','distinct_payment_type_count']] = delivered_order_items[['payment_record_count','distinct_payment_type_count']].astype('int')

In [70]:
#validate these two columns by checking:Their datatypes, Their missing-value counts, Their minimum values
print(f"payment_record_count & distinct_payment_type_count data type:\n{delivered_order_items[['payment_record_count','distinct_payment_type_count']].dtypes}")

print(f"payment_record_count & distinct_payment_type_count missing value count:\n{delivered_order_items[['payment_record_count','distinct_payment_type_count']].isna().sum()}")
print(f"minimum of payment_record_count : {min(delivered_order_items['payment_record_count'])}")
print(f"minimum of distinct_payment_type_count : {min(delivered_order_items['distinct_payment_type_count'])}")

payment_record_count & distinct_payment_type_count data type:
payment_record_count           int64
distinct_payment_type_count    int64
dtype: object
payment_record_count & distinct_payment_type_count missing value count:
payment_record_count           0
distinct_payment_type_count    0
dtype: int64
minimum of payment_record_count : 0
minimum of distinct_payment_type_count : 0


In [71]:
# Label the dominant payment type for orders without Payment records as missing
delivered_order_items['dominant_payment_type'] = delivered_order_items['dominant_payment_type'].fillna("missing")

In [72]:
#validate
print(f"Missing-value count: {delivered_order_items['dominant_payment_type'].isna().sum()}")
print(f"Number of rows labeled missing: {(delivered_order_items['dominant_payment_type']== 'missing').sum()}")

Missing-value count: 0
Number of rows labeled missing: 3


In [73]:
#Inspect the payment-quality flag after handling unmatched Payment records.
print(f"flag_payment_quality_exception data type:{delivered_order_items['flag_payment_quality_exception'].dtype}")

flag_payment_quality_exception data type:object


In [74]:
#Inspect missing Review measures after constructing the delivered dataset.
reviews_order_summary_columns_except_orderid = ['review_record_count', 'average_review_score', 'review_title_count', 'review_message_count', 'average_review_response_duration', 'flag_review_response_over_30_days']
delivered_order_items[reviews_order_summary_columns_except_orderid].isna().sum()

review_record_count                  827
average_review_score                 827
review_title_count                   827
review_message_count                 827
average_review_response_duration     827
flag_review_response_over_30_days    827
dtype: int64

In [75]:
#Flag delivered order-item rows whose order has no Review record.
flag_missing_review_record = delivered_order_items['review_record_count'].isna()
delivered_order_items['flag_missing_review_record'] = flag_missing_review_record

print(f"flag_missing_review_record data type: {delivered_order_items['flag_missing_review_record'].dtype}")
print(f" flag_missing_review_record True count: {int(delivered_order_items['flag_missing_review_record'].sum())}")

missing_flag_missing_review_records = delivered_order_items[delivered_order_items['flag_missing_review_record'] == True]
print(f"distinct order_id values among flagged rows: {missing_flag_missing_review_records['order_id'].nunique()}")


flag_missing_review_record data type: bool
 flag_missing_review_record True count: 827
distinct order_id values among flagged rows: 646


In [76]:
#Set missing review counts to zero and restore their integer data types.
delivered_order_items[['review_record_count','review_title_count','review_message_count']] = delivered_order_items[['review_record_count','review_title_count','review_message_count']].fillna(0)
delivered_order_items[['review_record_count','review_title_count','review_message_count']] = delivered_order_items[['review_record_count','review_title_count','review_message_count']].astype('int')

In [77]:
#Validate the cleaned review-count columns.
print(f"updated review count columns data types:\n{delivered_order_items[['review_record_count','review_title_count','review_message_count']].dtypes}")
print(f"updated review count columns missing value counts:\n{delivered_order_items[['review_record_count','review_title_count','review_message_count']].isna().sum()}")
print(f"Minimum value of review_record_count: {min(delivered_order_items['review_record_count'])}")
print(f"Minimum value of review_title_count: {min(delivered_order_items['review_title_count'])}")
print(f"Minimum value of review_message_count: {min(delivered_order_items['review_message_count'])}")

updated review count columns data types:
review_record_count     int64
review_title_count      int64
review_message_count    int64
dtype: object
updated review count columns missing value counts:
review_record_count     0
review_title_count      0
review_message_count    0
dtype: int64
Minimum value of review_record_count: 0
Minimum value of review_title_count: 0
Minimum value of review_message_count: 0


In [78]:
#Identify rows with a Review record but no usable review score.
review_record_missing_score = delivered_order_items[(delivered_order_items['flag_missing_review_record'] == False) & (delivered_order_items['average_review_score'].isna())]
review_record_missing_score

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,flag_future_shipping_limit_date,customer_id,order_status,...,flag_payment_quality_exception,dominant_payment_type,review_record_count,average_review_score,review_title_count,review_message_count,average_review_response_duration,flag_review_response_over_30_days,flag_missing_payment_record,flag_missing_review_record


In [79]:
#Confirm that missing review scores correspond exactly to missing Review records.
missing_review_score = delivered_order_items['average_review_score'].isna()

missing_review_record = delivered_order_items['flag_missing_review_record']

(missing_review_score == missing_review_record).all()

np.True_

In [80]:
#Inspect missing review-response measures for orders without Review records.
print(f"Missing value for average_review_response_duration: {int(delivered_order_items['average_review_response_duration'].isna().sum())} ")
print(f"Missing value for flag_review_response_over_30_days: {int(delivered_order_items['flag_review_response_over_30_days'].isna().sum())} ")

Missing value for average_review_response_duration: 827 
Missing value for flag_review_response_over_30_days: 827 


In [81]:
#Confirm that missing review-response measures correspond exactly to missing Review records.
missing_review_record = (delivered_order_items['flag_missing_review_record'] == True)

missing_average_review_response_duration_n_response_over_30_days = (delivered_order_items['average_review_response_duration'].isna()) & (delivered_order_items['flag_review_response_over_30_days'].isna())


(missing_review_record == missing_average_review_response_duration_n_response_over_30_days).all()

np.True_

In [82]:
#Validate remaining missing Payment and Review measures after handling unmatched records.
payment_summary_columns_except_orderid =['total_payment_value', 'payment_record_count', 'distinct_payment_type_count', 'maximum_payment_installments', 'flag_payment_quality_exception', 'dominant_payment_type']
reviews_order_summary_columns_except_orderid  
print(f"remaining missing Payments measures after handling unmatched records: {delivered_order_items[payment_summary_columns_except_orderid].isna().sum()}")
print(f"remaining missing Reviews measures after handling unmatched records: {delivered_order_items[reviews_order_summary_columns_except_orderid].isna().sum()}")

remaining missing Payments measures after handling unmatched records: total_payment_value               3
payment_record_count              0
distinct_payment_type_count       0
maximum_payment_installments      3
flag_payment_quality_exception    3
dominant_payment_type             0
dtype: int64
remaining missing Reviews measures after handling unmatched records: review_record_count                    0
average_review_score                 827
review_title_count                     0
review_message_count                   0
average_review_response_duration     827
flag_review_response_over_30_days    827
dtype: int64


## 7. Create Analytical Features

New features are derived from the cleaned delivered-order-item dataset to support delivery-performance, freight-burden, time-trend, customer-behavior, and product-performance analyses. Missing source information is preserved so that unavailable values are not treated as valid measurements.


### 7.1 Create Delivery-Performance Features

Delivery duration is calculated from purchase to actual customer delivery. The difference between actual and estimated delivery is measured both precisely and by calendar day. Calendar-day differences are used to classify orders as early, on time, late, or missing because estimated delivery timestamps contain only midnight time values.


In [83]:
#Calculate the delivery duration in days from purchase to customer delivery.
delivered_order_items['delivery_duration_days'] = delivered_order_items['order_delivered_customer_date'] - delivered_order_items['order_purchase_timestamp']
delivered_order_items['delivery_duration_days']

0         7 days 14:44:46
1        16 days 05:11:18
2         7 days 22:45:45
3         6 days 03:32:04
4        25 days 02:44:40
               ...       
110192   17 days 08:59:34
110193    9 days 10:05:09
110194    4 days 19:14:26
110195    1 days 22:56:41
110196    5 days 00:56:08
Name: delivery_duration_days, Length: 110197, dtype: timedelta64[us]

In [84]:
#convert the complete timedelta—including hours, minutes, and seconds—into numeric days and save it back to delivery_duration_days.
delivered_order_items['delivery_duration_days'] = delivered_order_items['delivery_duration_days'] / pd.Timedelta(days=1)


In [85]:
#Validate the delivery-duration feature and identify unavailable durations.

print(f"delivery-duration datatype: {delivered_order_items['delivery_duration_days'].dtype}")

missing_delivery_duration_days_records = delivered_order_items[delivered_order_items['delivery_duration_days'].isna()]
print(f"Count of order_id missing delivery-duration: {int((delivered_order_items['order_id'].isin(missing_delivery_duration_days_records['order_id'])).sum())}")

print(f"Number of distinct order_id missing delivery-duration: {missing_delivery_duration_days_records['order_id'].nunique()}")
print(f"Minimum of deivery-duration: {min(delivered_order_items['delivery_duration_days'])}")
print(f"Maximum of deivery-duration: {max(delivered_order_items['delivery_duration_days'])}")
print(f"Number of negative deivery-duration: {int((delivered_order_items['delivery_duration_days']<0).sum())}")

delivery-duration datatype: float64
Count of order_id missing delivery-duration: 8
Number of distinct order_id missing delivery-duration: 8
Minimum of deivery-duration: 0.5334143518518518
Maximum of deivery-duration: 209.6286111111111
Number of negative deivery-duration: 0


In [86]:
#Calculate the difference in days between actual and estimated delivery.
delivered_order_items['delivery_difference_days'] = delivered_order_items['order_delivered_customer_date'] - delivered_order_items['order_estimated_delivery_date']

In [87]:
#Convert this column into days preserving fractional values.
delivered_order_items['delivery_difference_days'] = delivered_order_items['delivery_difference_days'] / pd.Timedelta(days=1)

In [88]:
#Validate the actual-versus-estimated delivery difference.

print(f"delivery_difference_days datatype: {delivered_order_items['delivery_difference_days'].dtype}")

missing_delivery_difference_days_records = delivered_order_items[delivered_order_items['delivery_difference_days'].isna()]
print(f"Count of order_id missing delivery-duration: {int((delivered_order_items['order_id'].isin(missing_delivery_difference_days_records['order_id'])).sum())}")

print(f"Number of distinct order_id missing delivery_difference_days: {missing_delivery_difference_days_records['order_id'].nunique()}")
print(f"Minimum of delivery_difference_days: {min(delivered_order_items['delivery_difference_days'])}")
print(f"Maximum of delivery_difference_days: {max(delivered_order_items['delivery_difference_days'])}")
print(f"Number of negative valus-delivery_difference_days: {int((delivered_order_items['delivery_difference_days'] < 0).sum())}")
print(f"Number of Zero values-delivery_difference_days: {int((delivered_order_items['delivery_difference_days'] == 0).sum())}")
print(f"Number of possitive values-delivery_difference_days: {int((delivered_order_items['delivery_difference_days']>0).sum())}")

delivery_difference_days datatype: float64
Count of order_id missing delivery-duration: 8
Number of distinct order_id missing delivery_difference_days: 8
Minimum of delivery_difference_days: -146.0161226851852
Maximum of delivery_difference_days: 188.97508101851852
Number of negative valus-delivery_difference_days: 101475
Number of Zero values-delivery_difference_days: 0
Number of possitive values-delivery_difference_days: 8714


In [89]:
#Inspect estimated-delivery time values before creating calendar-day delivery categories.

estimated_delivery_times = delivered_order_items['order_estimated_delivery_date'].dt.time

print(f"Number of unique estimated-delivery times: {estimated_delivery_times.nunique()}")

print(f"Unique estimated-delivery times: {estimated_delivery_times.unique()}")

Number of unique estimated-delivery times: 1
Unique estimated-delivery times: [datetime.time(0, 0)]


In [90]:
#Calculate the calendar-day difference between actual and estimated delivery
actual_delivery_date = delivered_order_items['order_delivered_customer_date'].dt.date
estimated_delivery_date = delivered_order_items['order_estimated_delivery_date'].dt.date

delivered_order_items['delivery_calendar_difference_days'] = actual_delivery_date - estimated_delivery_date

delivered_order_items['delivery_calendar_difference_days']  = pd.to_timedelta(delivered_order_items['delivery_calendar_difference_days'])

delivered_order_items['delivery_calendar_difference_days'] = delivered_order_items['delivery_calendar_difference_days'].dt.days

In [91]:
#Validate the calendar-day delivery-difference feature.
missing_val_count = int(delivered_order_items['delivery_calendar_difference_days'].isna().sum())
negative_val_count = int((delivered_order_items['delivery_calendar_difference_days'] < 0).sum())
zero_val_count = int((delivered_order_items['delivery_calendar_difference_days'] == 0).sum())
positive_val_count = int((delivered_order_items['delivery_calendar_difference_days'] > 0).sum())
sum_of_all = missing_val_count + negative_val_count + zero_val_count + positive_val_count
print(f"delivery_calendar_difference_days data-type: {delivered_order_items['delivery_calendar_difference_days'].dtype}")
print(f"delivery_calendar_difference_days missing value count: {missing_val_count}")
print(f"Minimum of delivery_calendar_difference_days: {int(min(delivered_order_items['delivery_calendar_difference_days']))}")
print(f"Maximum of delivery_calendar_difference_days: {int(max(delivered_order_items['delivery_calendar_difference_days']))}")
print(f"Negative values count of delivery_calendar_difference_days: {negative_val_count}")
print(f"Zero values count of delivery_calendar_difference_days: {zero_val_count}")
print(f"Possitive values count of delivery_calendar_difference_days: {positive_val_count}")
print(f"sum of missing, negative, zero, possitive values is: {sum_of_all}")


delivery_calendar_difference_days data-type: float64
delivery_calendar_difference_days missing value count: 8
Minimum of delivery_calendar_difference_days: -147
Maximum of delivery_calendar_difference_days: 188
Negative values count of delivery_calendar_difference_days: 101475
Zero values count of delivery_calendar_difference_days: 1450
Possitive values count of delivery_calendar_difference_days: 7264
sum of missing, negative, zero, possitive values is: 110197


In [92]:
delivered_order_items['delivery_calendar_difference_days'] 


0         -9.0
1         -3.0
2        -14.0
3         -6.0
4        -16.0
          ... 
110192    -8.0
110193    -9.0
110194   -13.0
110195    -9.0
110196   -14.0
Name: delivery_calendar_difference_days, Length: 110197, dtype: float64

In [93]:
#Classify deliveries as early, on time, late, or missing

delivery_conditions = [ delivered_order_items['delivery_calendar_difference_days'] < 0, 
                        delivered_order_items['delivery_calendar_difference_days'] == 0,
                        delivered_order_items['delivery_calendar_difference_days'] > 0]

delivery_labels = ['early', 'on_time', 'late']

delivered_order_items['delivery_status'] = np.select(delivery_conditions, delivery_labels,default='missing')
delivered_order_items['delivery_status']

0         early
1         early
2         early
3         early
4         early
          ...  
110192    early
110193    early
110194    early
110195    early
110196    early
Name: delivery_status, Length: 110197, dtype: str

In [94]:
#Validate the delivery-status classification.
print(f"delivery status data type: {delivered_order_items['delivery_status'].dtype}")
print(f"Missing delivery status count: {int(delivered_order_items['delivery_status'].isna().sum())}\n")
print(f"Unique delivery status labels:\n{delivered_order_items['delivery_status'].unique()}\n")
print(f"frequency of each delivery status labels:\n{delivered_order_items['delivery_status'].value_counts()}")

delivery status data type: str
Missing delivery status count: 0

Unique delivery status labels:
<ArrowStringArray>
['early', 'on_time', 'late', 'missing']
Length: 4, dtype: str

frequency of each delivery status labels:
delivery_status
early      101475
late         7264
on_time      1450
missing         8
Name: count, dtype: int64


#### Delivery-Feature Validation Summary

The delivery features were created successfully without changing the delivered order-item grain. Delivery duration ranges from approximately 0.53 to 209.63 days, with no negative durations. Calendar-date comparison classified 101,475 item rows as early, 1,450 as on time, and 7,264 as late. Eight rows remain labeled as missing because their actual customer delivery timestamps are unavailable. Order-level delivery rates will be calculated later using distinct orders to avoid giving additional weight to multi-item orders.


### 7.2 Create the Freight-Burden Feature

Freight burden measures shipping cost relative to the merchandise price of each delivered item. Product prices must first be validated because missing, zero, or negative prices would make the percentage unavailable or invalid.


In [95]:
#Validate product prices before calculating freight burden.
print(f"Price - Missing value count: {int(delivered_order_items['price'].isna().sum())}")
print(f"Price - Zero-value count: {int((delivered_order_items['price']==0).sum())}")
print(f"Price - Negative value count: {int((delivered_order_items['price']<0).sum())}")
print(f"Price - Minimum value: {min(delivered_order_items['price'])}")

Price - Missing value count: 0
Price - Zero-value count: 0
Price - Negative value count: 0
Price - Minimum value: 0.85


In [96]:
# Validate freight values before calculating freight burden.
print(f"Freight - Missing value count: {int(delivered_order_items['freight_value'].isna().sum())}")
print(f"Freight - Zero-value count: {int((delivered_order_items['freight_value']==0).sum())}")
print(f"Freight - Negative value count: {int((delivered_order_items['freight_value']<0).sum())}")
print(f"Freight - Minimum value: {min(delivered_order_items['freight_value'])}")
print(f"Freight - Maximum value: {max(delivered_order_items['freight_value'])}")

Freight - Missing value count: 0
Freight - Zero-value count: 381
Freight - Negative value count: 0
Freight - Minimum value: 0.0
Freight - Maximum value: 409.68


In [97]:
#Calculate freight cost as a percentage of each delivered item’s price.
freight_burden_percentage = (delivered_order_items['freight_value'] / delivered_order_items['price']) *100
delivered_order_items['freight_burden_percentage'] = freight_burden_percentage

In [98]:
#Validate the freight-burden percentage and inspect extreme values.
print(f"freight_burden_percentage data type: {delivered_order_items['freight_burden_percentage'].dtype}")
print(f"freight_burden_percentage missing value count: {int(delivered_order_items['freight_burden_percentage'].isna().sum())}")
print(f"freight_burden_percentage Infinite value count: {int(np.isinf(delivered_order_items['freight_burden_percentage']).sum())}")
print(f"Minimum freight_burden_percentage: {min(delivered_order_items['freight_burden_percentage'])}")
print(f"Maximum freight_burden_percentage: {max(delivered_order_items['freight_burden_percentage'])}")
print(f"Number of rows with 0%: {int((delivered_order_items['freight_burden_percentage']==0).sum())}")
print(f"Number of freight_burden_percentage value greater than 100%: {int((delivered_order_items['freight_burden_percentage'] > 100).sum())}")


freight_burden_percentage data type: float64
freight_burden_percentage missing value count: 0
freight_burden_percentage Infinite value count: 0
Minimum freight_burden_percentage: 0.0
Maximum freight_burden_percentage: 2623.5294117647063
Number of rows with 0%: 381
Number of freight_burden_percentage value greater than 100%: 4008


In [99]:
#Inspect items with the highest freight burden percentages.
high_freight_burden_percentage = delivered_order_items[['order_id', 'order_item_id', 'product_id', 'price', 'freight_value', 'freight_burden_percentage']].sort_values(by='freight_burden_percentage', ascending=False)
high_freight_burden_percentage.head(10)

,order_id,order_item_id,product_id,price,freight_value,freight_burden_percentage
85212,c5bdd8ef3c0ec420232e668302179113,2,8a3254bee785a526d548a81a9bc3c9be,0.85,22.30,2623.529412
47542,6e864b3f0ec71031117ad4cf46b7f2a1,1,8a3254bee785a526d548a81a9bc3c9be,0.85,18.23,2144.705882
27030,3ee6513ae7ea23bdfab5b9ab60bffcb5,1,8a3254bee785a526d548a81a9bc3c9be,0.85,18.23,2144.705882
56045,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000
56055,8272b63d03f5f79c56e9e4120aec44ef,19,270516a3f41dc035aa87d220228f844c,1.20,7.89,657.500000
56041,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000
56042,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000
56040,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000
56038,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000
56037,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,1.20,7.89,657.500000


#### Freight-Burden Validation Summary

The freight-burden feature was created for all 110,197 delivered item rows without missing or infinite values. A total of 381 items have a freight burden of 0%, representing zero-cost shipping, while 4,008 items have freight costs exceeding their merchandise price. The highest percentages are associated with very low-priced products and are retained as meaningful business observations rather than treated as data errors.


### 7.3 Create Time-Based Features

Time-based features are derived from the order purchase timestamp to support monthly trends, growth calculations, seasonality analysis, and comparisons across years, quarters, months, and weekdays.


In [100]:
# Create the order purchase year-month feature for monthly trend analysis.

delivered_order_items['order_purchase_month'] = delivered_order_items['order_purchase_timestamp'].dt.to_period('M')


In [101]:
# Validate the order purchase year-month feature and date coverage.
print(f"order_purchase_month datatype: {delivered_order_items['order_purchase_month'].dtype}")
print(f"order_purchase_month missing value count: {int(delivered_order_items['order_purchase_month'].isna().sum())}")
print(f"Earliest purchase month: {min(delivered_order_items['order_purchase_month'])}")
print(f"Latest purchase month: {max(delivered_order_items['order_purchase_month'])}")
print(f"Number of unique purchase months: {delivered_order_items['order_purchase_month'].nunique()}")

order_purchase_month datatype: period[M]
order_purchase_month missing value count: 0
Earliest purchase month: 2016-09
Latest purchase month: 2018-08
Number of unique purchase months: 23


In [102]:
# Identify missing months within the delivered-order purchase period.
complete_month_range = pd.period_range(min(delivered_order_items['order_purchase_month']), max(delivered_order_items['order_purchase_month']), freq='M')
complete_month_range

PeriodIndex(['2016-09', '2016-10', '2016-11', '2016-12', '2017-01', '2017-02',
             '2017-03', '2017-04', '2017-05', '2017-06', '2017-07', '2017-08',
             '2017-09', '2017-10', '2017-11', '2017-12', '2018-01', '2018-02',
             '2018-03', '2018-04', '2018-05', '2018-06', '2018-07', '2018-08'],
            dtype='period[M]')

In [103]:
observed_purchase_months = delivered_order_items['order_purchase_month'].unique()
observed_purchase_months

<PeriodArray>
['2017-09', '2017-04', '2018-01', '2018-08', '2017-02', '2017-05', '2017-12',
 '2018-07', '2018-03', '2018-02', '2018-04', '2018-05', '2017-08', '2017-11',
 '2017-03', '2018-06', '2017-10', '2017-01', '2017-06', '2017-07', '2016-10',
 '2016-12', '2016-09']
Length: 23, dtype: period[M]

In [104]:
missing_purchase_month = complete_month_range.difference(observed_purchase_months)
missing_purchase_month

PeriodIndex(['2016-11'], dtype='period[M]')

In [105]:
# Extract the order purchase year for annual comparisons.
delivered_order_items['order_purchase_year'] = delivered_order_items['order_purchase_timestamp'].dt.year
delivered_order_items['order_purchase_year']

0         2017
1         2017
2         2018
3         2018
4         2017
          ... 
110192    2018
110193    2018
110194    2017
110195    2017
110196    2018
Name: order_purchase_year, Length: 110197, dtype: int32

In [106]:
# Extract the order purchase quarter for seasonal comparisons.
delivered_order_items['order_purchase_quarter'] = delivered_order_items['order_purchase_timestamp'].dt.quarter
delivered_order_items['order_purchase_quarter'] 

0         3
1         2
2         1
3         3
4         1
         ..
110192    2
110193    3
110194    4
110195    3
110196    2
Name: order_purchase_quarter, Length: 110197, dtype: int32

In [107]:
# Extract the purchase month number for month-of-year comparisons.
delivered_order_items['order_purchase_month_number'] = delivered_order_items['order_purchase_timestamp'].dt.month
delivered_order_items['order_purchase_month_number']

0          9
1          4
2          1
3          8
4          2
          ..
110192     4
110193     7
110194    10
110195     8
110196     6
Name: order_purchase_month_number, Length: 110197, dtype: int32

In [108]:
# Extract the order purchase weekday for weekly pattern analysis.
delivered_order_items['order_purchase_weekday'] = delivered_order_items['order_purchase_timestamp'].dt.day_name()
delivered_order_items['order_purchase_weekday']

0         Wednesday
1         Wednesday
2            Sunday
3         Wednesday
4          Saturday
            ...    
110192       Monday
110193     Saturday
110194       Monday
110195       Monday
110196     Saturday
Name: order_purchase_weekday, Length: 110197, dtype: str

In [109]:
# Extract the weekday number to preserve chronological weekday ordering.
delivered_order_items['order_purchase_weekday_number'] = delivered_order_items['order_purchase_timestamp'].dt.weekday
delivered_order_items['order_purchase_weekday_number']

0         2
1         2
2         6
3         2
4         5
         ..
110192    0
110193    5
110194    0
110195    0
110196    5
Name: order_purchase_weekday_number, Length: 110197, dtype: int32

In [110]:
# Validate all time-based analytical features.
time_based_colmns = ['order_purchase_month', 'order_purchase_year', 'order_purchase_quarter', 'order_purchase_month_number', 'order_purchase_weekday', 'order_purchase_weekday_number']

print(f"data-type: \n{delivered_order_items[time_based_colmns].dtypes}")

data-type: 
order_purchase_month             period[M]
order_purchase_year                  int32
order_purchase_quarter               int32
order_purchase_month_number          int32
order_purchase_weekday                 str
order_purchase_weekday_number        int32
dtype: object


In [111]:
print(f"missing value count:\n{delivered_order_items[time_based_colmns].isna().sum()}")

missing value count:
order_purchase_month             0
order_purchase_year              0
order_purchase_quarter           0
order_purchase_month_number      0
order_purchase_weekday           0
order_purchase_weekday_number    0
dtype: int64


In [112]:
print(f"Unique years: {delivered_order_items['order_purchase_year'].unique()}")

Unique years: [2017 2018 2016]


In [113]:
print(f"Quarter range from {min(delivered_order_items['order_purchase_quarter'])} to {max(delivered_order_items['order_purchase_quarter'])}")

Quarter range from 1 to 4


In [114]:
print(f"Month number range from {min(delivered_order_items['order_purchase_month_number'])} to {max(delivered_order_items['order_purchase_month_number'])}")

Month number range from 1 to 12


In [115]:
print(f"Unique weekday names:\n{delivered_order_items['order_purchase_weekday'].unique()}")

Unique weekday names:
<ArrowStringArray>
['Wednesday', 'Sunday', 'Saturday', 'Monday', 'Friday', 'Tuesday', 'Thursday']
Length: 7, dtype: str


In [116]:
print(f"Weekday number range from {min(delivered_order_items['order_purchase_weekday_number'])} to {max(delivered_order_items['order_purchase_weekday_number'])}")

Weekday number range from 0 to 6


#### Time-Feature Validation Summary

Six time-based features were created successfully from the order purchase timestamp without introducing missing values. The delivered-order purchase period spans September 2016 through August 2018 and includes 23 observed months. November 2016 has no delivered-order purchases and will be inserted with zero activity when the monthly KPI table is created.


## 8. Final Analysis-Ready Dataset Validation

The completed delivered-order-item dataset is validated to confirm that feature engineering preserved its intended grain, row count, order coverage, status scope, key uniqueness, missing-value meaning, and analytical data types.


In [117]:
# Validate the final dataset structure and delivered order-item grain.
print(f"final shape of delivered_order_items table: {delivered_order_items.shape}")

final shape of delivered_order_items table: (110197, 74)


In [118]:
print(f"Distinct order_id count: {delivered_order_items['order_id'].nunique()}")

Distinct order_id count: 96478


In [119]:
print(f"Unique order_status values: {delivered_order_items['order_status'].unique()}")

Unique order_status values: <ArrowStringArray>
['delivered']
Length: 1, dtype: str


In [120]:
print(f"Duplicated order_id-order_item_id combinations: {delivered_order_items[['order_id', 'order_item_id']].duplicated().sum()}")

Duplicated order_id-order_item_id combinations: 0


In [121]:
print(f"duplicated column names: {delivered_order_items.columns.duplicated().sum()}")

duplicated column names: 0


In [122]:
# Review all remaining missing values in the final analysis-ready dataset.
delivered_order_items_missing_values = delivered_order_items.isna().sum()
delivered_order_items_missing_values_greaterthan_0 = delivered_order_items_missing_values[delivered_order_items_missing_values>0]
delivered_order_items_missing_values_greaterthan_0

order_approved_at                      15
order_delivered_carrier_date            2
order_delivered_customer_date           8
product_category_name                1537
product_name_length                  1537
product_description_length           1537
product_photos_qty                   1537
product_weight_g                       18
product_length_cm                      18
product_height_cm                      18
product_width_cm                       18
product_category_name_english        1559
geolocation_reference_state           249
total_payment_value                     3
maximum_payment_installments            3
flag_payment_quality_exception          3
average_review_score                  827
average_review_response_duration      827
flag_review_response_over_30_days     827
delivery_duration_days                  8
delivery_difference_days                8
delivery_calendar_difference_days       8
dtype: int64

In [123]:
# Inspect nullable payment and review column datatypes before final conversion.
nullable_payment_review_columns = ['maximum_payment_installments', 'flag_payment_quality_exception', 'flag_review_response_over_30_days']

print(f"data types of nullable payment and reviews columns:\n {delivered_order_items[nullable_payment_review_columns].dtypes}")

data types of nullable payment and reviews columns:
 maximum_payment_installments         float64
flag_payment_quality_exception        object
flag_review_response_over_30_days     object
dtype: object


In [124]:
# Convert nullable payment and review fields to appropriate analytical data types.
delivered_order_items['maximum_payment_installments'] = delivered_order_items['maximum_payment_installments'].astype('Int64')
delivered_order_items['flag_payment_quality_exception'] = delivered_order_items['flag_payment_quality_exception'].astype('boolean')
delivered_order_items['flag_review_response_over_30_days'] = delivered_order_items['flag_review_response_over_30_days'].astype('boolean')

In [125]:
# Validate nullable payment and review field conversions.
print(f"data types of nullable payment and reviews columns after conversion:\n {delivered_order_items[nullable_payment_review_columns].dtypes}")

data types of nullable payment and reviews columns after conversion:
 maximum_payment_installments           Int64
flag_payment_quality_exception       boolean
flag_review_response_over_30_days    boolean
dtype: object


In [126]:
print(f"Missing value counts of nullable payment and reviews columns:\n {delivered_order_items[nullable_payment_review_columns].isna().sum()}")

Missing value counts of nullable payment and reviews columns:
 maximum_payment_installments           3
flag_payment_quality_exception         3
flag_review_response_over_30_days    827
dtype: int64


In [127]:
delivered_order_items['flag_payment_quality_exception'].value_counts(dropna=False)

flag_payment_quality_exception
False    110102
True         92
<NA>          3
Name: count, dtype: Int64

In [128]:
delivered_order_items['flag_review_response_over_30_days'].value_counts(dropna=False)

flag_review_response_over_30_days
False    108646
<NA>        827
True        724
Name: count, dtype: Int64

In [129]:
# Convert the calendar-day delivery difference to a nullable integer datatype.
delivered_order_items['delivery_calendar_difference_days'] = delivered_order_items['delivery_calendar_difference_days'].astype('Int64')

print(f"delivery_calender_difference_days data type: {delivered_order_items['delivery_calendar_difference_days'].dtype} ")
print(f"delivery_calender_difference_days missing value count: {delivered_order_items['delivery_calendar_difference_days'].isna().sum()}")

delivery_calender_difference_days data type: Int64 
delivery_calender_difference_days missing value count: 8


In [130]:
# Review the final dataset datatype distribution before saving.
delivered_order_items_data_type = delivered_order_items.dtypes
delivered_order_items_data_type.value_counts()

str                19
bool               18
float64            14
int64               7
datetime64[us]      6
int32               4
Int64               2
boolean             2
timedelta64[us]     1
period[M]           1
Name: count, dtype: int64

### Final Dataset Validation Summary

The analysis-ready dataset contains 110,197 delivered order-item records across 96,478 distinct orders and 74 columns. The original order-item grain is preserved, with no duplicated order–item keys or duplicated column names. All records remain within the delivered-order scope. Derived delivery, freight-burden, and time-based features were validated successfully. Remaining missing values reflect documented source-data limitations or unmatched Payment and Review records and have not been replaced with misleading values.


## 9. Save the Analysis-Ready Dataset

The validated delivered-order-item dataset is saved in a format that preserves datetime, timedelta, period, nullable integer, and nullable Boolean data types. A separate CSV version may also be created later for dashboard or cross-platform use.


In [131]:
# Save the validated analysis-ready dataset in Parquet format.
delivered_order_items.to_parquet(Processed_Data_Path/"delivered_order_items_analysis_ready.parquet", index=False)

In [132]:
# Reload and validate the saved Parquet dataset.
delivered_order_items_reload = pd.read_parquet(Processed_Data_Path/"delivered_order_items_analysis_ready.parquet")

In [133]:
print(f"delivered_order_items_reload shape: {delivered_order_items_reload.shape}")

delivered_order_items_reload shape: (110197, 74)


In [134]:
print(f"delivered_order_items_reload distinct order_id : {delivered_order_items_reload['order_id'].nunique()}")

delivered_order_items_reload distinct order_id : 96478


In [138]:
print(f"delivered_order_items_reload duplicated order_id-order_item_id combination : {int(delivered_order_items_reload[['order_id', 'order_item_id']].duplicated().sum())}")

delivered_order_items_reload duplicated order_id-order_item_id combination : 0


In [153]:
delivered_order_items_missing_value_count = delivered_order_items.isna().sum()
delivered_order_items_missing_value_count_greaterthan_0 = delivered_order_items_missing_value_count[delivered_order_items_missing_value_count>0]

delivered_order_items_reload_missing_value_count = delivered_order_items_reload.isna().sum()
delivered_order_items_reload_missing_value_count_greaterthan_0 = delivered_order_items_reload_missing_value_count[delivered_order_items_reload_missing_value_count>0]

missing_values_summary = pd.DataFrame({"delivered_order_items": delivered_order_items_missing_value_count_greaterthan_0, "delivered_order_items_reload":delivered_order_items_reload_missing_value_count_greaterthan_0})
missing_values_summary

,delivered_order_items,delivered_order_items_reload
order_approved_at,15,15
order_delivered_carrier_date,2,2
order_delivered_customer_date,8,8
product_category_name,1537,1537
product_name_length,1537,1537
product_description_length,1537,1537
product_photos_qty,1537,1537
product_weight_g,18,18
product_length_cm,18,18
product_height_cm,18,18


In [152]:
# data types of
reloaded_column = ['order_purchase_timestamp',
                   'order_purchase_month',
                   'maximum_payment_installments', 
                   'flag_payment_quality_exception',
                   'flag_review_response_over_30_days',
                   'delivery_calendar_difference_days']

print(f"data type: {delivered_order_items_reload[reloaded_column].dtypes}")

data type: order_purchase_timestamp             datetime64[us]
order_purchase_month                      period[M]
maximum_payment_installments                  Int64
flag_payment_quality_exception              boolean
flag_review_response_over_30_days           boolean
delivery_calendar_difference_days             Int64
dtype: object


### Saved Dataset Validation Summary

The saved Parquet dataset was reloaded successfully with the expected 110,197 rows and 74 columns. It retains 96,478 distinct delivered orders, contains no duplicated order–item keys, and preserves all reviewed analytical data types. Missing-value counts are identical to the in-memory dataset, confirming that no records, values, or missing-value information were altered during export and reload.
